<a href="https://colab.research.google.com/github/mykisaacs/MyMassivelyParallelCode/blob/main/notebooks/VecAddParallel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Running CUDA C / C++ (`.cu`) Code in Colab

Google Colab provides GPUs and pre-installed CUDA toolkits, making it an excellent environment for developing and running CUDA C and C++ applications. The standard extension is `.cu`.

Let's go through the steps:
1.  **Verify CUDA Installation**
2.  **Create a CUDA C / C++ Source File (`.cu`)**
3.  **Compile the CUDA Code using `nvcc`**
4.  **Run the Executable**

### 1. Verify nvcc CUDA Installation

First let's check if `nvcc` the CUDA Compiler is available and see it's version. It's usually in Colab pre-installed.

In [ ]:
import subprocess

try:
  # Check nvcc version
  result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True, check=True)
  print(result.stdout)
except FileNotFoundError:
  print("nvcc not found. CUDA Toolkit might not be installed or not in PATH.")
except subprocess.CalledProcessError as e:
  print(f"Error checking nvcc version: {e}\n{e.stderr}")

### 2. Create a CUDA C / C++ Source File (.cu)

We'll create a simple CUDA C program named vec_add_parallel.cu to show vector addition kernel functionality.

In [ ]:
%% writefile vec_add_parallel.cu

#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

// Compute vector sum C_h = A_h + B_h on CPU
void vecAdd(float* A_h, float* B_h, float* C_h, int n) {
  for (int i = 0; i < n; ++i) {
    C_h[i] = A_h[i] + B_h[i];
  }
}

__host__
void vecAdd_Parallel(float* A, float* B, float* C, int n) {
  float *A_d,  *B_d, *C_d;
  int size = n* sizeof(float) ;

  // Part 1: Allocate device memory for A, B and C
  // Below wrapped in checkResult method that checks returned flag
  checkResult(cudaMalloc((void**)&A_d, size));
  checkResult(cudaMalloc((void**)&B_d, size));
  checkResult(cudaMalloc((void**)&C_d, size)) ;

  // Copy A and B to device memorycheckResult
  cudaMemcpy(A_d, A_h, size, cudaMemcpyHostToDevice) ;
  cudaMemcpy(B_d, B_h, size, cudaMemcpyHostToDevice) ;

  // Part 2: Call kernel - to launch a grid of threads
  // to perform vector addition
  vecAddKernel<<<ceil(n/256.0), 256>>>(A_d, B_d, C_d, n)  ;

  // Part 3: Copy C from the device memory
  cudaMemcpy(C_h, C_d, size, cudaMemcpyHostToDevice) ;
  // Free device vectors
   cudaFree(A_d);
   cudaFree(B_d);
   cudaFree(C_d);

}

__global__
void vecAddKernel(float* A, float* B, float* C, int n) {
  int i = threadIdx.x + blockDim.x * blockIdx.x ;
  if (i < n) {
    C[i] = A[i] + B[i] ;
  }
}



void checkResult(cudaError_t err) {
  if (err != cudaSuccess) {
    printf("%s in %s at line %d\n", cudaGetErrorString(err), __FILE__, __LINE__) ;
    exit(EXIT_FAILURE) ;
  }
}


int main() {
  int N;

  // 1. I/O to read vector size
  printf("Enter the number of elements: ");
  if (scanf("%d", &N) != 1 || N <= 0) {
    printf("Invalid size.\n:");
    return 1 ;
  }

  // 2. Memory allocation for host arrays A, B and C
  float* A = (float*)malloc(N * sizeof(float)) ;
  float* B = (float*)malloc(N * sizeof(float)) ;
  float* C = (float*)malloc(N * sizeof(float)) ;

  if (A==NULL || B==NULL || C == NULL) {
    printf("Memory allocation failed.\n") ;
    return 1 ;
  }

  // 3. I/O to read elements for A and B
  printf("Enter %d elements for vector A:\n", N) ;
  for (int i = 0; i < N; ++i) {
    scanf("%f", &A[i]);
  }

  printf("Enter %d elements for vector B:\n", N) ;
  for (int i = 0; i < N; ++i) {
    scanf("%f", &B[i]);
  }

  // 4. Call the vector addition function
  vecAdd(A, B, C, N) ;

  // 5. Output the results
  printf("Result of Vector Addition\n") ;

  for (int i = 0 ; i < N ; ++i) {
    printf("%.2f + %.2f = %.2f\n", A[i], B[i], C[i]) ;
  }

  // 6. Free allocated memory
  free(A) ;
  free(B) ;
  free(C) ;

  return 0 ;
}


### 3. Compile the CUDA Code using `nvcc`

Now we'll use the `nvcc` compiler to compile our `vec_add_parallel.cu` file into an executable. We'll name the output executable `vec_add_parallel`.

In [ ]:
!nvcc vec_add_parallel.cu -o vec_add_parallel

### 4. Run the Executable

Finally execute the compiled program.

In [ ]:
!.vec_add_parallel